In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

class GPTDatasetV1 (Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))


    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size = 4,max_length = 256, stride=128,shuffle=True, drop_last=True, num_workers = 0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)
    dataloader = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers = num_workers)
    return dataloader


with open ("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

max_length = 4
dataloader = create_dataloader_v1(raw_text,batch_size=8,max_length=max_length,stride=1,shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token input IDs:\n", inputs)
print("\n Inputs shape:\n",inputs.shape)

Token input IDs:
 tensor([[   40,   367,  2885,  1464],
        [  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257]])

 Inputs shape:
 torch.Size([8, 4])


In [2]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim) # 对 input token id 做embedding。一个token id 对应一个256 向量 但是这是初始化的

token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [3]:
print(token_embeddings[2][3])

tensor([ 0.1336, -1.2718,  0.7731, -0.0423, -0.4468,  0.4092,  0.2254,  1.3564,
        -1.2937,  0.4887,  0.7238,  0.7217,  0.1158, -0.6798,  0.5716, -1.2809,
         1.0347, -0.1876,  0.3423,  0.1153,  2.0289, -0.6533,  0.7670, -0.8813,
         0.7530,  1.6201,  1.3097,  0.8263, -0.5155,  0.3427, -1.7759, -0.4218,
        -0.6066,  1.8994,  1.6675,  0.1862, -0.7738, -0.5402,  0.1575,  0.0892,
        -1.2216,  0.9729, -1.1082,  1.5550,  1.4681, -0.1392, -0.8754, -0.1110,
        -1.4175,  0.7887,  0.5885,  0.2865, -0.7564,  1.5911, -0.4423,  1.0757,
         0.0963,  0.0202, -0.3128,  0.0160, -0.7746,  1.6116, -0.5389, -1.3066,
         0.9651,  0.5622, -0.0494,  1.7644,  0.7270, -0.1982, -2.0605,  1.3372,
        -1.0059,  0.7668, -0.0778,  1.3354,  0.8689, -1.4778,  0.2971, -0.2507,
         0.7789,  2.0795,  0.1643, -1.5601, -0.6521, -0.8831, -0.9120, -0.6228,
         1.1405, -0.4094, -0.6240, -1.1161, -0.3794,  1.1918, -2.9035, -0.7736,
        -1.3476,  1.0763,  0.9593, -0.04

In [4]:
print(token_embeddings[3][2])

tensor([ 0.1336, -1.2718,  0.7731, -0.0423, -0.4468,  0.4092,  0.2254,  1.3564,
        -1.2937,  0.4887,  0.7238,  0.7217,  0.1158, -0.6798,  0.5716, -1.2809,
         1.0347, -0.1876,  0.3423,  0.1153,  2.0289, -0.6533,  0.7670, -0.8813,
         0.7530,  1.6201,  1.3097,  0.8263, -0.5155,  0.3427, -1.7759, -0.4218,
        -0.6066,  1.8994,  1.6675,  0.1862, -0.7738, -0.5402,  0.1575,  0.0892,
        -1.2216,  0.9729, -1.1082,  1.5550,  1.4681, -0.1392, -0.8754, -0.1110,
        -1.4175,  0.7887,  0.5885,  0.2865, -0.7564,  1.5911, -0.4423,  1.0757,
         0.0963,  0.0202, -0.3128,  0.0160, -0.7746,  1.6116, -0.5389, -1.3066,
         0.9651,  0.5622, -0.0494,  1.7644,  0.7270, -0.1982, -2.0605,  1.3372,
        -1.0059,  0.7668, -0.0778,  1.3354,  0.8689, -1.4778,  0.2971, -0.2507,
         0.7789,  2.0795,  0.1643, -1.5601, -0.6521, -0.8831, -0.9120, -0.6228,
         1.1405, -0.4094, -0.6240, -1.1161, -0.3794,  1.1918, -2.9035, -0.7736,
        -1.3476,  1.0763,  0.9593, -0.04

In [8]:
context_length= max_length
pos_embedding_layer = torch.nn.Embedding(context_length,output_dim) # 4*256
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [9]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
